# Trích đặc trưng EfficientNet-B0 cho SVM và CNN-LSTM

**BTL Khai phá dữ liệu – HK261.** 

Notebook này chạy mạng **EfficientNet-B0** (học trước trên ImageNet, đóng băng) trên **mọi** video **một lần duy nhất** và lưu đặc trưng ra file. Notebook SVM và CNN-LSTM sau đó chỉ đọc các file này, nên chạy trong vài phút và dùng **chung** một đầu vào để so sánh công bằng.

Mỗi video đi qua đúng đường xử lý của `preprocess.py`: bỏ khung đơn sắc → lấy đều 16 khung → (chỉ với bản tăng cường) cắt đoạn thời gian, cắt vùng ảnh, đổi màu → chuẩn hoá ImageNet. Mỗi khung cho ra một vector 1.280 chiều.

| File đầu ra | Kích thước mảng | Dùng cho |
|---|---|---|
| `feat_train.npy` | (2772, 16, 1280) | SVM; validation của CNN-LSTM |
| `feat_train_aug.npy` | (2772, 5, 16, 1280) – 5 bản tăng cường mỗi video | huấn luyện CNN-LSTM giai đoạn A |
| `feat_public.npy` | (1630, 16, 1280) | dự đoán public test |
| `feat_private.npy` | (2859, 16, 1280) | dự đoán private test |

Thứ tự dòng của `feat_train*` khớp `manifest_clean.csv`, của `feat_public` / `feat_private` khớp `test_manifest.csv` (đã chép kèm vào output). Lưu dạng float16.

In [1]:
import os, sys, json, time, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **k: x

SEED = 42
N_AUG = 5
BATCH_VIDEOS = int(os.environ.get('BATCH_VIDEOS', '8'))   # 8 video x 16 khung = 128 ảnh mỗi lượt qua GPU
NUM_WORKERS = int(os.environ.get('NUM_WORKERS', min(4, os.cpu_count() or 1)))

LIMIT = int(os.environ.get('LIMIT', '0'))                 # > 0: chỉ lấy LIMIT video mỗi tập
PRETRAINED = os.environ.get('PRETRAINED', '1') == '1'     # 0: không tải trọng số ImageNet

ON_KAGGLE = Path('/kaggle/input').exists()
ON_COLAB = 'google.colab' in sys.modules
if ON_KAGGLE:
    SEARCH_ROOTS = [Path('/kaggle/input')]
    OUT = Path('/kaggle/working')
elif ON_COLAB:
    SEARCH_ROOTS = [Path('/content')]
    OUT = Path('/content/features')
else:
    ROOT = Path(os.environ.get('PROJECT_ROOT', Path.cwd().parent))
    SEARCH_ROOTS = [ROOT / 'data', ROOT / 'processing']
    OUT = ROOT / 'models' / 'features'
OUT = Path(os.environ.get('OUT_DIR', OUT))
OUT.mkdir(parents=True, exist_ok=True)


def find_file(name):
    for root in SEARCH_ROOTS:
        hits = sorted(root.rglob(name)) if root.exists() else []
        if hits:
            return hits[0]
    raise FileNotFoundError(f'Không tìm thấy {name} trong {SEARCH_ROOTS}')


LABEL_PKL = find_file('label_mapping.pkl')
DATASET = LABEL_PKL.parent                     
PP_FILE = find_file('preprocess.py')
MANIFEST = find_file('manifest_clean.csv')
TEST_MANIFEST = find_file('test_manifest.csv')
CLASS_WEIGHTS = find_file('class_weights.csv')

sys.path.insert(0, str(PP_FILE.parent))
import preprocess as pp

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dataset      :', DATASET)
print('preprocess.py:', PP_FILE)
print('Output       :', OUT)
print('Thiết bị     :', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '')
print('Số tiến trình đọc video:', NUM_WORKERS, '| LIMIT =', LIMIT, '| PRETRAINED =', PRETRAINED)

Dataset      : /kaggle/input/datasets/ngdieuvy/vsl-sign-language/dataset/dataset
preprocess.py: /kaggle/input/datasets/ngdieuvy/vsl-sign-language/preprocess.py
Output       : /kaggle/working
Thiết bị     : cuda Tesla T4
Số tiến trình đọc video: 4 | LIMIT = 0 | PRETRAINED = True


## 1. Đọc manifest và kiểm tra file video

Video train lấy theo `manifest_clean.csv` (đã làm sạch ở bước tiền xử lý), kèm cột `valid_idx` là chỉ số các khung không đơn sắc đã tính sẵn. Video test lấy theo `test_manifest.csv`; khung đơn sắc của video test được tính lại khi đọc. Tên thư mục lớp trên đĩa có thể ở dạng Unicode NFC hoặc NFD, nên đường dẫn được tra qua `pp.class_dirs`.

In [2]:
train = pd.read_csv(MANIFEST)
test = pd.read_csv(TEST_MANIFEST)
public = test[test.split == 'public_test'].reset_index(drop=True)
private = test[test.split == 'private_test'].reset_index(drop=True)
if LIMIT:
    train, public, private = train.head(LIMIT), public.head(LIMIT), private.head(LIMIT)

dirs = pp.class_dirs(DATASET / 'train')
train_paths = [DATASET / 'train' / dirs[pp.nfc(c)] / f for c, f in zip(train.cls, train.file)]
public_paths = [DATASET / 'public_test' / f for f in public.file]
private_paths = [DATASET / 'private_test' / f for f in private.file]

for name, paths in [('train', train_paths), ('public', public_paths), ('private', private_paths)]:
    missing = [p for p in paths if not p.exists()]
    assert not missing, f'{name}: thiếu {len(missing)} file, ví dụ {missing[:3]}'
    print(f'{name:8s}: {len(paths)} video, đủ file')
if not LIMIT:
    assert (len(train), len(public), len(private)) == (2772, 1630, 2859)

train   : 2772 video, đủ file
public  : 1630 video, đủ file
private : 2859 video, đủ file


## 2. Mô hình trích đặc trưng

EfficientNet-B0 bỏ lớp phân loại cuối (`classifier = Identity`), nên mỗi khung 224×224 cho ra vector 1.280 chiều sau global average pooling. Mô hình ở chế độ `eval` (tắt dropout, BatchNorm dùng thống kê đã học) và không tính gradient. Trên GPU dùng số thực 16 bit (`autocast`) để nhanh gấp khoảng đôi.

In [3]:
weights = EfficientNet_B0_Weights.IMAGENET1K_V1 if PRETRAINED else None
model = efficientnet_b0(weights=weights)
model.classifier = torch.nn.Identity()
model = model.eval().to(DEVICE).to(memory_format=torch.channels_last)
FEAT_DIM = 1280
T = pp.TARGET_FRAMES
STORE_DTYPE = np.float16 if PRETRAINED else np.float32

with torch.no_grad():
    probe = model(torch.zeros(2, 3, 224, 224, device=DEVICE).to(memory_format=torch.channels_last))
assert probe.shape == (2, FEAT_DIM), probe.shape
print('EfficientNet-B0:', sum(p.numel() for p in model.parameters()) / 1e6, 'triệu tham số | đầu ra', tuple(probe.shape))

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 181MB/s]


EfficientNet-B0: 4.007548 triệu tham số | đầu ra (2, 1280)


## 3. Đọc video song song và trích đặc trưng

Mỗi phần tử của `ClipSet` là một video (hoặc một bản tăng cường của một video). Bản tăng cường thứ `k` của video thứ `j` dùng bộ sinh số ngẫu nhiên khởi tạo từ `(SEED, j, k)`, nên chạy lại notebook cho ra đúng các bản tăng cường cũ. `DataLoader` đọc video trên nhiều tiến trình CPU trong lúc GPU tính đặc trưng.

In [4]:
AUG = pp.VideoAugmentation()


class ClipSet(Dataset):
    def __init__(self, paths, valid=None, n_copies=0):
        self.paths, self.valid, self.n_copies = list(paths), valid, n_copies

    def __len__(self):
        return len(self.paths) * max(1, self.n_copies)

    def __getitem__(self, i):
        j, k = divmod(i, self.n_copies) if self.n_copies else (i, 0)
        v = self.valid[j] if self.valid is not None else None
        if self.n_copies:
            clip = pp.load_clip(self.paths[j], valid_idx=v, aug=AUG, rng=np.random.default_rng([SEED, j, k]))
        else:
            clip = pp.load_clip(self.paths[j], valid_idx=v)
        return torch.from_numpy(pp.normalize(clip)), i


@torch.no_grad()
def extract(ds, desc):
    out = np.zeros((len(ds), T, FEAT_DIM), dtype=STORE_DTYPE)
    dl = DataLoader(ds, batch_size=BATCH_VIDEOS, num_workers=NUM_WORKERS,
                    pin_memory=DEVICE.type == 'cuda', persistent_workers=False)
    t0 = time.time()
    for x, idx in tqdm(dl, desc=desc):
        b = x.shape[0]
        x = x.to(DEVICE, non_blocking=True).flatten(0, 1).contiguous(memory_format=torch.channels_last)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=DEVICE.type == 'cuda'):
            f = model(x)
        out[idx.numpy()] = f.float().reshape(b, T, FEAT_DIM).cpu().numpy().astype(STORE_DTYPE)
    print(f'{desc}: {len(ds)} clip trong {time.time() - t0:.0f}s')
    return out


timing = {}
t = time.time(); feat_train = extract(ClipSet(train_paths, train.valid_idx.tolist()), 'train'); timing['train'] = time.time() - t
t = time.time(); feat_public = extract(ClipSet(public_paths), 'public'); timing['public'] = time.time() - t
t = time.time(); feat_private = extract(ClipSet(private_paths), 'private'); timing['private'] = time.time() - t
t = time.time()
feat_aug = extract(ClipSet(train_paths, train.valid_idx.tolist(), n_copies=N_AUG), 'train_aug')
feat_aug = feat_aug.reshape(len(train_paths), N_AUG, T, FEAT_DIM)
timing['train_aug'] = time.time() - t
print({k: round(v) for k, v in timing.items()})

train:   0%|          | 0/347 [00:00<?, ?it/s]

train: 2772 clip trong 89s


public:   0%|          | 0/204 [00:00<?, ?it/s]

public: 1630 clip trong 116s


private:   0%|          | 0/358 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x780ee286f420>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
Exception ignored in:     self._shutdown_workers()Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x780ee286f420>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1655, in _shutdown_workers
    self._pin_memory_thread.join()
  File "/usr/lib/python3.13/threading.py", line 1088, in join
    raise RuntimeError("cannot join current thread")
RuntimeError: cannot join current thread

<function _MultiProcessingDataLoaderIter.__del__ at 0x780ee286f420>  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, i

private: 2859 clip trong 198s


train_aug:   0%|          | 0/1733 [00:00<?, ?it/s]

train_aug: 13860 clip trong 674s
{'train': 89, 'public': 116, 'private': 198, 'train_aug': 674}


## 4. Kiểm tra nhanh

- Kích thước mảng đúng, không có giá trị NaN/vô cực, không có video nào toàn số 0 (nghĩa là ô đó chưa được ghi).
- **Thử phân loại đơn giản:** lấy trung bình 16 khung, tính vector trung bình (tâm) của mỗi lớp trên phần train (fold ≠ 0), rồi gán mỗi video fold 0 vào lớp có tâm gần nhất theo cosine. Đoán bừa chỉ được khoảng 1%; nếu kết quả cao hơn hẳn thì đặc trưng có ý nghĩa. Đây chỉ là kiểm tra, không phải kết quả mô hình.

In [5]:
for name, a in [('train', feat_train), ('train_aug', feat_aug), ('public', feat_public), ('private', feat_private)]:
    a32 = a.astype(np.float32)
    zero_rows = int((np.abs(a32).reshape(len(a32), -1).sum(1) == 0).sum())
    print(f'{name:9s} {str(a.shape):24s} NaN/inf: {int((~np.isfinite(a32)).sum())} | video toàn 0: {zero_rows} '
          f'| mean {a32.mean():.3f} | std {a32.std():.3f}')
    assert np.isfinite(a32).all() and zero_rows == 0

from sklearn.metrics import accuracy_score, f1_score

pooled = feat_train.astype(np.float32).mean(1)
pooled /= np.linalg.norm(pooled, axis=1, keepdims=True) + 1e-8
tr, va = (train.fold != 0).values, (train.fold == 0).values
if va.sum() and len(np.unique(train.label_idx[tr])) > 1:
    classes = np.unique(train.label_idx[tr])
    centroids = np.stack([pooled[tr][train.label_idx[tr] == c].mean(0) for c in classes])
    pred = classes[(pooled[va] @ centroids.T).argmax(1)]
    y = train.label_idx[va].values
    print(f'Tâm lớp gần nhất trên fold 0: accuracy {accuracy_score(y, pred):.3f} | Macro F1 '
          f'{f1_score(y, pred, average="macro"):.3f} (đoán bừa ~0.010)')
else:
    print('Bỏ qua thử phân loại (chạy thử với quá ít video)')

train     (2772, 16, 1280)         NaN/inf: 0 | video toàn 0: 0 | mean 0.152 | std 0.394
train_aug (2772, 5, 16, 1280)      NaN/inf: 0 | video toàn 0: 0 | mean 0.146 | std 0.392
public    (1630, 16, 1280)         NaN/inf: 0 | video toàn 0: 0 | mean 0.153 | std 0.398
private   (2859, 16, 1280)         NaN/inf: 0 | video toàn 0: 0 | mean 0.145 | std 0.393
Tâm lớp gần nhất trên fold 0: accuracy 0.085 | Macro F1 0.091 (đoán bừa ~0.010)


## 5. Lưu kết quả

Ngoài 4 file đặc trưng, notebook chép kèm manifest, trọng số lớp, `preprocess.py` và danh sách tên lớp, để notebook SVM / CNN-LSTM chỉ cần gắn **output của notebook này** là đủ. `label_names.json` giữ tên lớp **đúng như khoá gốc trong `label_mapping.pkl`** (dạng Unicode NFD), dùng khi ghi file nộp AIChallenge.

In [6]:
np.save(OUT / 'feat_train.npy', feat_train)
np.save(OUT / 'feat_train_aug.npy', feat_aug)
np.save(OUT / 'feat_public.npy', feat_public)
np.save(OUT / 'feat_private.npy', feat_private)

train.to_csv(OUT / 'manifest_clean.csv', index=False)
pd.concat([public, private]).to_csv(OUT / 'test_manifest.csv', index=False)
shutil.copy(CLASS_WEIGHTS, OUT / 'class_weights.csv')
shutil.copy(PP_FILE, OUT / 'preprocess.py')

import pickle
with open(LABEL_PKL, 'rb') as f:
    raw = pickle.load(f)
label_names = [None] * len(raw)
for k, v in raw.items():
    label_names[int(v)] = k
assert None not in label_names
with open(OUT / 'label_names.json', 'w', encoding='utf-8') as f:
    json.dump(label_names, f, ensure_ascii=False)

meta = {'backbone': 'efficientnet_b0', 'weights': 'IMAGENET1K_V1' if PRETRAINED else None, 'feat_dim': FEAT_DIM,
        'T': T, 'n_aug': N_AUG, 'seed': SEED, 'dtype': np.dtype(STORE_DTYPE).name, 'limit': LIMIT,
        'n_train': len(train), 'n_public': len(public), 'n_private': len(private),
        'seconds': {k: round(v) for k, v in timing.items()}, 'device': str(DEVICE),
        'torch': torch.__version__}
with open(OUT / 'meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

for p in sorted(OUT.iterdir()):
    if p.is_file():
        print(f'{p.name:24s} {p.stat().st_size / 1e6:8.1f} MB')

__notebook__.ipynb            0.0 MB
class_weights.csv             0.0 MB
feat_private.npy            117.1 MB
feat_public.npy              66.8 MB
feat_train.npy              113.5 MB
feat_train_aug.npy          567.7 MB
label_names.json              0.0 MB
manifest_clean.csv            0.5 MB
meta.json                     0.0 MB
preprocess.py                 0.0 MB
test_manifest.csv             0.3 MB
